In [1]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader

class MNISTDataset(Dataset):
    def __init__(self, csv_file, is_train=True):
        self.data = pd.read_csv(csv_file)
        self.is_train = is_train
        if self.is_train:
            self.y = self.data['label'].values
            self.X = self.data.drop('label', axis=1).values / 255.0
        else:
            self.X = self.data.values / 255.0

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.is_train:
            label = int(self.y[idx])
            y_one_hot = torch.zeros(10)
            y_one_hot[label] = 1.0
            return torch.tensor(self.X[idx], dtype=torch.float32), y_one_hot
        return torch.tensor(self.X[idx], dtype=torch.float32)

def flatten_and_transpose(X: torch.Tensor):
    if len(X.shape) > 2:
        X = torch.reshape(X, (X.shape[0], -1))
    return X.T

def init_params(n_features, layers_size):
    params = {}
    for i in range(len(layers_size)):
        if i == 0:
            width = n_features
        else:
            width = layers_size[i - 1]
        height = layers_size[i]
        w = torch.rand(size=(height, width), dtype=torch.float) * torch.sqrt(torch.tensor(2.0 / width))
        b = torch.zeros(size=(height, 1), dtype=torch.float)
        params[f'W{i + 1}'] = w
        params[f'b{i + 1}'] = b
    return params

def forward_prop(input_data, params):
    n_layers = len(params) // 2
    z = input_data
    cache = {}
    cache['A0'] = input_data
    for i in range(1, n_layers + 1):
        if i != n_layers:
            tmp_z = torch.matmul(params[f'W{i}'], z) + params[f'b{i}']
            tmp_a = torch.mul(tmp_z, (tmp_z > 0).to(torch.float))
            z = tmp_a
            cache[f'Z{i}'] = tmp_z
            cache[f'A{i}'] = tmp_a
        else:
            tmp_z = torch.matmul(params[f'W{i}'], z) + params[f'b{i}']
            tmp_z_stable = tmp_z - torch.max(tmp_z, dim=0, keepdim=True).values
            tmp_a = torch.exp(tmp_z_stable) / torch.exp(tmp_z_stable).sum(dim=0)
            z = tmp_a
            cache[f'Z{i}'] = tmp_z_stable
            cache[f'A{i}'] = tmp_a
    return z, cache

def backward_prop(A, Y, params, cache):
    grads = {}
    L = len(params) // 2
    m = A.size()[1]
    grads[f'dZ{L}'] = A - Y
    grads[f'dW{L}'] = torch.matmul(A - Y, cache[f'A{L - 1}'].T) / m
    grads[f'db{L}'] = (A - Y).sum(axis=1, keepdim=True) / m

    for i in range(L - 1, 0, -1):
        grads[f'dZ{i}'] = torch.mul(
            torch.matmul(params[f'W{i + 1}'].T, grads[f'dZ{i + 1}']),
            (cache[f'Z{i}'] > 0).to(torch.float)
        )
        grads[f'dW{i}'] = torch.matmul(grads[f'dZ{i}'], cache[f'A{i - 1}'].T) / m
        grads[f'db{i}'] = grads[f'dZ{i}'].sum(axis=1, keepdim=True) / m

    return grads

def update_params(params, grads, lr=0.01):
    L = len(params) // 2
    for i in range(1, L + 1):
        params[f'W{i}'] -= lr * grads[f'dW{i}']
        params[f'b{i}'] -= lr * grads[f'db{i}']
    return params

def train(train_csv_path, n_features=28 * 28, epochs=10, lr=0.05, batch_size=64):
    train_dataset = MNISTDataset(train_csv_path, is_train=True)
    layers_size = [16, 16, 10]
    cost_per_epochs = []
    params = init_params(n_features, layers_size)

    for e in range(1, epochs + 1):
        train_data_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        n_batch = len(train_data_loader)
        cost = 0
        
        for _, (X, y) in enumerate(train_data_loader):
            ft_X = flatten_and_transpose(X)
            A, cache = forward_prop(ft_X, params)
            grads = backward_prop(A, y.T, params, cache)
            params = update_params(params, grads, lr=lr)
            cost += -torch.sum(torch.mul(y.T, torch.log(A + 1e-9))) / n_batch
        
        cost_per_epochs.append(cost.item())
        print(f'Epoch [{e}/{epochs}]: cost = {cost.item()}')
    print('Done.')

    return params, cost_per_epochs

def predict(params, X_test):
    predict_probs, _ = forward_prop(X_test, params)
    label = torch.argmax(predict_probs, dim=0)
    return label

def submit(params, test_csv_path, output_path='submission.csv'):
    test_dataset = MNISTDataset(test_csv_path, is_train=False)
    test_data_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)
    
    predictions = []
    for X in test_data_loader:
        ft_X = flatten_and_transpose(X)
        batch_preds = predict(params, ft_X)
        predictions.extend(batch_preds.tolist())
        
    submission = pd.DataFrame({
        'ImageId': range(1, len(predictions) + 1),
        'Label': predictions
    })
    submission.to_csv(output_path, index=False)

In [2]:
train_path = 'data\\lab8\\train.csv'
test_path = 'data\\lab8\\test.csv'
submission_path = 'data\\lab8\\submission.csv'

In [3]:
params, cost_per_epochs = train(
    train_csv_path=train_path,
    n_features=28 * 28,
    epochs=15,
    lr=0.05,
    batch_size=64
)

Epoch [1/15]: cost = 67.36663818359375
Epoch [2/15]: cost = 23.2752685546875
Epoch [3/15]: cost = 18.921260833740234
Epoch [4/15]: cost = 16.038555145263672
Epoch [5/15]: cost = 14.296735763549805
Epoch [6/15]: cost = 13.185647964477539
Epoch [7/15]: cost = 12.314671516418457
Epoch [8/15]: cost = 11.665066719055176
Epoch [9/15]: cost = 11.096806526184082
Epoch [10/15]: cost = 10.555624961853027
Epoch [11/15]: cost = 10.270818710327148
Epoch [12/15]: cost = 9.756538391113281
Epoch [13/15]: cost = 9.43536376953125
Epoch [14/15]: cost = 9.081381797790527
Epoch [15/15]: cost = 8.825878143310547
Done.


In [4]:
submit(
    params=params,
    test_csv_path=test_path,
    output_path=submission_path
)
print('Submit successfully.')

Submit successfully.
